## 사전 고정 결정사항 (DESIGN.md §2)

| 결정 | 내용 |
|------|------|
| Q17_13 분류 | **Formal 유지** — 문항이 "보유(possession)", 근로감독 확인 가능 |
| 결과변수 | **ra_score** (Q14/Q15 위험성평가 실질성) |
| 건설업 | **이질성 검증 집단** — 불지지를 "예측된 발견"으로 프레이밍 |
| 표준화 | **그룹 composite, 0~1 min-max** — F·S 동일 범위 |

---

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "factor_analyzer", "scikit-learn<1.6", "pingouin",
                "statsmodels", "scipy", "matplotlib", "seaborn", "openpyxl"],
               capture_output=True)
print("패키지 준비 완료")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

BASE   = "../data/raw/"
OUTDIR = "output/"

print("임포트 완료")

---
## Phase 1: 데이터 전처리
`로드 → 변수 존재 검증 → 무응답 NaN → 이진 재코딩 → 이상치 → ra_score → 결측률`

In [ ]:
# 1-1. 원본 로드
df_mfg = pd.read_csv(BASE + "제10차_산업안전보건_실태조사_raw_data_제조업_230824.csv")
df_svc = pd.read_csv(BASE + "제10차_산업안전보건_실태조사_raw_data_서비스업_230824.csv")
df_con = pd.read_csv(BASE + "제10차_산업안전보건_실태조사_raw_data_건설업_230824.CSV")

print(f"제조업:   {df_mfg.shape[0]:,} 행 × {df_mfg.shape[1]} 열")
print(f"서비스업: {df_svc.shape[0]:,} 행 × {df_svc.shape[1]} 열")
print(f"건설업:   {df_con.shape[0]:,} 행 × {df_con.shape[1]} 열")

In [ ]:
# 1-2. 변수 존재 검증 — 코드북 표기 vs 실데이터 열명 대조
print("=" * 60)
print("[제조업] F/S 관련 변수 실제 열명")
print("-" * 60)
for prefix in ['Q6', 'Q8', 'Q9', 'Q10', 'Q11', 'Q13', 'Q15', 'Q16_2', 'Q17', 'Q18', 'Q22']:
    cols = sorted(df_mfg.filter(like=prefix).columns.tolist())
    if cols:
        print(f"  {prefix}: {cols}")

print()
print("=" * 60)
print("[건설업] F/S 관련 변수 실제 열명")
print("-" * 60)
for prefix in ['Q6', 'Q8', 'Q9', 'Q10', 'Q12', 'Q14', 'Q15_2', 'Q16', 'Q17', 'Q21']:
    cols = sorted(df_con.filter(like=prefix).columns.tolist())
    if cols:
        print(f"  {prefix}: {cols}")

In [ ]:
# 1-3. 무응답 NaN 처리 함수
def apply_nan(df, cols, codes):
    """지정 열에서 무응답 코드를 NaN으로 교체."""
    existing = [c for c in cols if c in df.columns]
    df[existing] = df[existing].replace(codes, np.nan)
    return df

# ── 제조업 ──
# 리커트 계열 (9 → NaN)
likert_ms_prefixes = ('Q15_','Q16_','Q17_','Q18_','Q19_','Q22_')
likert_ms = [c for c in df_mfg.columns if c.startswith(likert_ms_prefixes)]
df_mfg = apply_nan(df_mfg, likert_ms, 9)
df_svc = apply_nan(df_svc, likert_ms, 9)

# 이진·범주 변수 (9 → NaN)
binary_ms = ['Q6', 'Q9_1', 'Q9_2', 'Q10', 'Q11', 'Q13_1', 'Q13_2']
df_mfg = apply_nan(df_mfg, binary_ms, 9)
df_svc = apply_nan(df_svc, binary_ms, 9)

# 인원수 (999, 99999 → NaN)
count_ms = ['Q8_1', 'Q8_2']
df_mfg = apply_nan(df_mfg, count_ms, [999, 99999])
df_svc = apply_nan(df_svc, count_ms, [999, 99999])

# ── 건설업 ──
likert_con_prefixes = ('Q14_','Q15_','Q16_','Q17_','Q18_','Q21_')
likert_con = [c for c in df_con.columns if c.startswith(likert_con_prefixes)]
df_con = apply_nan(df_con, likert_con, 9)

binary_con = ['Q6', 'Q9', 'Q10', 'Q12_1', 'Q12_2']
df_con = apply_nan(df_con, binary_con, 9)

count_con = ['Q8_1', 'Q8_2', 'Q8_3', 'Q8_4']
df_con = apply_nan(df_con, count_con, [999, 99999])

print("NaN 처리 완료")

In [ ]:
# 1-4. 이진 재코딩 (2=아니요 → 0) 및 범주 재코딩
def binary_recode(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = df[c].map({1: 1.0, 2: 0.0})
    return df

# 제조·서비스: 단순 이진
df_mfg = binary_recode(df_mfg, ['Q6', 'Q9_1', 'Q9_2', 'Q11', 'Q13_1', 'Q13_2'])
df_svc = binary_recode(df_svc, ['Q6', 'Q9_1', 'Q9_2', 'Q11', 'Q13_1', 'Q13_2'])

# Q10 (제조·서비스: 1=운영, 2=미운영, 3=모름)
for df in [df_mfg, df_svc]:
    if 'Q10' in df.columns:
        df['Q10'] = df['Q10'].map({1: 1.0, 2: 0.0, 3: np.nan})

# 건설: 단순 이진
df_con = binary_recode(df_con, ['Q6', 'Q9', 'Q12_1', 'Q12_2'])

# Q10 건설 (코드북: 1=운영, 2=협의체, 3=미운영, 4=모름 → 검증 후 적용)
# 우선 빈도 확인
print("건설 Q10 값 분포:", df_con['Q10'].value_counts(dropna=False).to_dict())
# 결과 확인 후 재코딩 적용
df_con['Q10'] = df_con['Q10'].map({1: 1.0, 2: 0.5, 3: 0.0, 4: np.nan})

print("이진 재코딩 완료")

In [ ]:
# 1-5. 이상치 윈저화 (Q8_1, Q8_2: 선임 인원 — 극단값 처리)
for col in ['Q8_1', 'Q8_2']:
    for df, name in [(df_mfg,'제조'), (df_svc,'서비스'), (df_con,'건설')]:
        if col in df.columns:
            p99 = df[col].quantile(0.99)
            n_clipped = (df[col] > p99).sum()
            df[col] = df[col].clip(upper=p99)
            if n_clipped > 0:
                print(f"  [{name}] {col}: 99퍼센타일={p99:.1f}, 윈저화 {n_clipped}건")

# 1-6. ra_score 생성 (위험성평가 실질성: 1→2, 2→1, 3→0)
ra_map = {1: 2, 2: 1, 3: 0}
df_mfg['ra_score'] = df_mfg['Q15'].map(ra_map)
df_svc['ra_score'] = df_svc['Q15'].map(ra_map)
df_con['ra_score'] = df_con['Q14'].map(ra_map)

# 1-7. 통제변수 생성
for df, w_col in [(df_mfg,'Q1_1'), (df_svc,'Q1_1')]:
    df['log_workers']  = np.log1p(df[w_col].replace([999,99999], np.nan))
    df['elder_rate']   = (df['Q1_2'] / df[w_col]).clip(0, 1)
    df['foreign_rate'] = (df['Q1_3'] / df[w_col]).clip(0, 1)
    df['female_rate']  = (df['Q1_4'] / df[w_col]).clip(0, 1)

df_con['log_workers']  = np.log1p(df_con['Q4_1'].replace([999,99999], np.nan))
df_con['elder_rate']   = (df_con['Q4_2'] / df_con['Q4_1']).clip(0, 1)
df_con['foreign_rate'] = (df_con['Q4_3'] / df_con['Q4_1']).clip(0, 1)
df_con['female_rate']  = (df_con['Q4_4'] / df_con['Q4_1']).clip(0, 1)

# 1-8. 제조+서비스 통합
df_mfg['industry'] = '제조'
df_svc['industry'] = '서비스'
df_ms = pd.concat([df_mfg, df_svc], ignore_index=True)
df_ms['ind_mfg'] = (df_ms['industry'] == '제조').astype(float)

print(f"제조+서비스 통합: {len(df_ms):,} 행")
print(f"건설:             {len(df_con):,} 행")

In [ ]:
# 1-9. 결측률 점검 (F/S 핵심 변수 — 10% 미만이어야 분석 진행)
key_f_ms = ['Q6','Q8_1','Q8_2','Q9_1','Q9_2','Q10','Q11','Q13_1','Q13_2','Q17_13']
key_s_ms = ['Q16_2_1','Q17_1','Q17_2','Q17_3','Q17_9','Q17_10',
            'Q17_12','Q17_15','Q17_16','Q17_17','Q18_1','Q22_2']
key_f_con = ['Q6','Q8_1','Q8_2','Q8_3','Q8_4','Q9','Q10','Q12_1','Q12_2']
key_s_con = ['Q15_2_1','Q16_1','Q16_2','Q16_3','Q16_9','Q16_10',
             'Q16_16','Q16_17','Q16_18','Q17_1','Q17_2','Q17_4','Q21_2']

def missing_report(df, f_keys, s_keys, label):
    print(f"\n{'='*55}")
    print(f"[{label}] 결측률 (%)")
    print(f"{'─'*55}")
    issues = []
    for tag, keys in [('F', f_keys), ('S', s_keys)]:
        avail = [c for c in keys if c in df.columns]
        miss  = df[avail].isnull().mean() * 100
        for c, v in miss.items():
            flag = ' ⚠️' if v >= 10 else ''
            print(f"  [{tag}] {c:15s}: {v:5.1f}%{flag}")
            if v >= 10:
                issues.append(c)
    if issues:
        print(f"\n  ⚠️  10% 이상 결측 변수: {issues}")
    else:
        print(f"\n  ✅  모든 변수 결측률 10% 미만")

missing_report(df_ms,  key_f_ms,  key_s_ms,  '제조+서비스')
missing_report(df_con, key_f_con, key_s_con, '건설')

---
## Phase 2: 측정 모형 타당성 검증
`내적 일관성(α) → EFA → 판별 타당도 → VIF`  
**⚠️ 이 단계가 통과되지 않으면 Phase 4로 진행하지 않는다.**

In [ ]:
# 2-1. S 변수 내적 일관성 (Cronbach α)
# S 변수는 동질 리커트 척도이므로 α 적용 가능
s_items_ms = ['Q17_1','Q17_2','Q17_3','Q17_9','Q17_10',
              'Q17_12','Q17_15','Q17_16','Q17_17',
              'Q18_1','Q18_2','Q18_3','Q18_4','Q22_2']
# Q16_2_* 는 위험성평가 참여 문항으로 별도 확인
s_items_ra = [c for c in df_ms.columns if c.startswith('Q16_2_')]

def cronbach_alpha(df, cols):
    """Cronbach α 계산 (listwise 결측 제거)."""
    data = df[cols].dropna()
    k = len(cols)
    if k < 2 or len(data) == 0:
        return np.nan
    item_var = data.var(axis=0, ddof=1).sum()
    total_var = data.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - item_var / total_var)

alpha_s_ms  = cronbach_alpha(df_ms, [c for c in s_items_ms if c in df_ms.columns])
alpha_ra_ms = cronbach_alpha(df_ms, [c for c in s_items_ra if c in df_ms.columns]) if s_items_ra else np.nan

s_items_con = ['Q16_1','Q16_2','Q16_3','Q16_9','Q16_10',
               'Q16_16','Q16_17','Q16_18','Q17_1','Q17_2','Q17_4']
s_items_ra_con = [c for c in df_con.columns if c.startswith('Q15_2_')]
alpha_s_con = cronbach_alpha(df_con, [c for c in s_items_con if c in df_con.columns])

print("Cronbach α (기준: α > 0.70)")
print(f"  [제조+서비스] S 핵심 문항: α = {alpha_s_ms:.3f}")
print(f"  [제조+서비스] 위험성평가 참여(Q16_2_*): α = {alpha_ra_ms:.3f}")
print(f"  [건설]        S 핵심 문항: α = {alpha_s_con:.3f}")
for val, lab in [(alpha_s_ms,'제조+서비스 S'),(alpha_s_con,'건설 S')]:
    status = '✅ 통과' if val > 0.70 else '⚠️ 기준 미달'
    print(f"  → [{lab}] {status}")

In [ ]:
# 2-2. EFA — S 변수 요인 구조 탐색 (제조+서비스)
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

# EFA 대상: S 핵심 리커트 문항 (Q17계열 + Q18계열)
efa_items_ms = [c for c in s_items_ms if c in df_ms.columns]
X_efa = df_ms[efa_items_ms].dropna()

kmo_val  = calculate_kmo(X_efa)[1]
bart_p   = calculate_bartlett_sphericity(X_efa)[1]
print(f"KMO: {kmo_val:.3f}  (기준 >0.60: {'✅' if kmo_val>0.60 else '⚠️'})")
print(f"Bartlett p: {bart_p:.4e}  (기준 <0.05: {'✅' if bart_p<0.05 else '⚠️'})")

# 2요인 EFA (Varimax)
fa2 = FactorAnalyzer(n_factors=2, method='principal', rotation='varimax')
fa2.fit(X_efa)
ev, _ = fa2.get_eigenvalues()
print(f"\n고유값(상위 4): {ev[:4].round(3)}")
print(f"2요인 누적 설명 분산: {(ev[:2]/ev.sum()*100).sum():.1f}%  → 각 {ev[0]/ev.sum()*100:.1f}%, {ev[1]/ev.sum()*100:.1f}%")

loadings_ms = pd.DataFrame(fa2.loadings_, index=efa_items_ms, columns=['F1','F2'])
loadings_ms['최대적재요인'] = loadings_ms.abs().idxmax(axis=1)
print("\n[제조+서비스] EFA 적재량 (|적재량| > 0.40 굵게 표시)")
print(loadings_ms.round(3).to_string())

# Q17_13 적재 위치 — 사전 결정과 불일치 시 각주 근거
if 'Q17_13' in loadings_ms.index:
    row = loadings_ms.loc['Q17_13']
    print(f"\n★ Q17_13 적재: F1={row['F1']:.3f}, F2={row['F2']:.3f} → {row['최대적재요인']}")
    print("  (사전 결정: Formal 유지 — 이론적 정의 우선)")

In [ ]:
# 2-3. 판별 타당도 + VIF 사전 점검 (임시 F/S 점수 사용)
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

def minmax_col(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn + 1e-10) if mx > mn else s * 0.0

# 임시 점수 (그룹 composite 전 대략 점수)
f_temp_ms = df_ms[['Q6','Q13_1','Q13_2']].apply(minmax_col).mean(axis=1)
s_temp_ms = df_ms[['Q17_1','Q17_2','Q17_3']].apply(minmax_col).mean(axis=1)

r_fs_ms = f_temp_ms.corr(s_temp_ms)
print(f"[제조+서비스] F-S 상관: r = {r_fs_ms:.3f}  (기준 r < 0.85: {'✅' if abs(r_fs_ms)<0.85 else '⚠️ 재검토'})")

f_temp_con = df_con[['Q6','Q12_1','Q12_2']].apply(minmax_col).mean(axis=1)
s_temp_con = df_con[['Q16_1','Q16_2','Q16_3']].apply(minmax_col).mean(axis=1)

r_fs_con = f_temp_con.corr(s_temp_con)
print(f"[건설]        F-S 상관: r = {r_fs_con:.3f}  (기준 r < 0.85: {'✅' if abs(r_fs_con)<0.85 else '⚠️ 재검토'})")

# VIF 점검
tmp = pd.DataFrame({'F': f_temp_ms, 'S': s_temp_ms}).dropna()
X_vif = sm.add_constant(tmp)
vif_vals = pd.Series(
    [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
    index=X_vif.columns
)
print(f"\n[제조+서비스] VIF — F: {vif_vals['F']:.2f}, S: {vif_vals['S']:.2f}")
print(f"  → VIF < 10: {'✅ 모형 B 안전' if vif_vals[['F','S']].max()<10 else '⚠️ 모형 A 의존 권장'}")

---
## Phase 3: F_score / S_score / SDI 산출 및 기술통계
`그룹 composite → 기술통계 → F vs S 산포도`

In [ ]:
# 3-1. 그룹 composite 함수 및 변수 목록 정의
def compute_group_composite(df, group_dict, label=''):
    """
    각 그룹 내 변수를 min-max 정규화 후 평균 → 그룹 점수
    그룹 점수들의 평균 → 최종 composite (0~1)
    """
    group_scores = {}
    for gname, cols in group_dict.items():
        avail = [c for c in cols if c in df.columns]
        if not avail:
            print(f"  ⚠️  [{label}] 그룹 {gname}: 변수 없음 {cols}")
            continue
        normed = df[avail].apply(minmax_col)
        group_scores[gname] = normed.mean(axis=1)
    if not group_scores:
        return pd.Series(np.nan, index=df.index)
    score = pd.DataFrame(group_scores).mean(axis=1)
    print(f"  [{label}] {len(group_scores)}그룹 | 범위 {score.min():.3f}~{score.max():.3f} | 평균 {score.mean():.3f}")
    return score

# ── 제조·서비스업 F/S 그룹 (DESIGN.md §4-1 확정 목록) ──
formal_groups_ms = {
    'F-M1': ['Q6'],
    'F-M2': ['Q8_1', 'Q8_2'],
    'F-M3': ['Q9_1', 'Q9_2'],
    'F-M4': ['Q10'],
    'F-M5': ['Q11'],
    'F-M6': ['Q13_1'],
    'F-M7': ['Q13_2'],
    'F-M8': ['Q17_13'],   # 경계 변수, Formal 고정
}
subst_groups_ms = {
    'S-M1': [c for c in df_ms.columns if c.startswith('Q16_2_')],
    'S-M2': ['Q17_1', 'Q17_2', 'Q17_3'],
    'S-M3': ['Q17_9', 'Q17_10'],
    'S-M4': ['Q17_12'],
    'S-M5': ['Q17_15', 'Q17_16', 'Q17_17'],
    'S-M6': ['Q18_1', 'Q18_2', 'Q18_3', 'Q18_4'],
    'S-M7': ['Q22_2'],
}

print(f"[제조+서비스] F 그룹 수: {len(formal_groups_ms)}, S 그룹 수: {len(subst_groups_ms)}")
print("  S-M1 실제 변수:", subst_groups_ms['S-M1'])

# ── 건설업 F/S 그룹 (DESIGN.md §4-2 확정 목록) ──
formal_groups_con = {
    'F-C1': ['Q6'],
    'F-C2': ['Q8_1', 'Q8_2'],
    'F-C3': ['Q8_3', 'Q8_4'],
    'F-C4': ['Q9'],
    'F-C5': ['Q10'],
    'F-C6': ['Q12_1'],
    'F-C7': ['Q12_2'],
}
subst_groups_con = {
    'S-C1': [c for c in df_con.columns if c.startswith('Q15_2_')],
    'S-C2': ['Q16_1', 'Q16_2', 'Q16_3'],
    'S-C3': ['Q16_9', 'Q16_10'],
    'S-C4': ['Q16_16', 'Q16_17', 'Q16_18'],
    'S-C5': ['Q17_1', 'Q17_2', 'Q17_4'],
    'S-C6': ['Q21_2'],
}

print(f"\n[건설] F 그룹 수: {len(formal_groups_con)}, S 그룹 수: {len(subst_groups_con)}")
print("  S-C1 실제 변수:", subst_groups_con['S-C1'])

In [ ]:
# 3-2. 점수 산출
print("F_score / S_score 산출")
df_ms['F_score'] = compute_group_composite(df_ms, formal_groups_ms, '제조+서비스 F')
df_ms['S_score'] = compute_group_composite(df_ms, subst_groups_ms,  '제조+서비스 S')
df_ms['SDI']     = df_ms['S_score'] - df_ms['F_score']

df_con['F_score'] = compute_group_composite(df_con, formal_groups_con, '건설 F')
df_con['S_score'] = compute_group_composite(df_con, subst_groups_con,  '건설 S')
df_con['SDI']     = df_con['S_score'] - df_con['F_score']

# 3-3. 기술통계 (비가중 + 가중 병기)
rows = []
for name, df in [('제조+서비스', df_ms), ('건설', df_con)]:
    for col in ['F_score', 'S_score', 'SDI']:
        s = df[col].dropna()
        w = df.loc[s.index, 'WT2']
        rows.append({
            '업종': name, '변수': col, 'n': len(s),
            '평균(비가중)': round(s.mean(), 4),
            '표준편차': round(s.std(), 4),
            'Q1': round(s.quantile(0.25), 4),
            '중앙값': round(s.median(), 4),
            'Q3': round(s.quantile(0.75), 4),
            '평균(가중WT2)': round(np.average(s, weights=w), 4),
        })

desc_df = pd.DataFrame(rows)
display(desc_df)
desc_df.to_csv(OUTDIR + 'descriptive_v5.csv', index=False, encoding='utf-8-sig')
print("\n[저장] descriptive_v5.csv")

# SDI < 0 비율
for name, df in [('제조+서비스', df_ms), ('건설', df_con)]:
    neg_pct = (df['SDI'] < 0).mean() * 100
    print(f"[{name}] SDI < 0 (형식주의 기업) 비율: {neg_pct:.1f}%")

In [ ]:
# 3-4. F vs S 산포도 (핵심 시각화)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

color_map = {'제조': '#1976D2', '서비스': '#FF8F00'}

# 왼쪽: 제조+서비스
ax = axes[0]
for ind, color in color_map.items():
    mask = df_ms['industry'] == ind
    ax.scatter(df_ms.loc[mask, 'F_score'], df_ms.loc[mask, 'S_score'],
               alpha=0.25, s=8, color=color, label=ind, rasterized=True)
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='F = S (SDI=0)')
ax.fill_between([0, 1], [0, 0], [0, 1], alpha=0.04, color='#1565C0', label='SDI > 0 (실질↑)')
ax.fill_between([0, 1], [0, 1], [1, 1], alpha=0.04, color='#B71C1C', label='SDI < 0 (형식주의)')
ax.set_xlabel('F_score (형식)', fontsize=11)
ax.set_ylabel('S_score (실질)', fontsize=11)
ax.set_title(f'제조+서비스 (n={len(df_ms):,})', fontsize=12)
ax.legend(fontsize=8, loc='upper left')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

# 오른쪽: 건설
ax = axes[1]
ax.scatter(df_con['F_score'], df_con['S_score'],
           alpha=0.25, s=8, color='#2E7D32', label='건설', rasterized=True)
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='F = S (SDI=0)')
ax.fill_between([0, 1], [0, 0], [0, 1], alpha=0.04, color='#1565C0')
ax.fill_between([0, 1], [0, 1], [1, 1], alpha=0.04, color='#B71C1C')
ax.set_xlabel('F_score (형식)', fontsize=11)
ax.set_ylabel('S_score (실질)', fontsize=11)
ax.set_title(f'건설 (n={len(df_con):,})', fontsize=12)
ax.legend(fontsize=8, loc='upper left')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.suptitle('F_score vs S_score 산포도\n(점선 아래: SDI<0 형식주의 영역)', fontsize=13)
plt.tight_layout()
plt.savefig(OUTDIR + 'fs_scatter_v5.png', dpi=150, bbox_inches='tight')
plt.show()
print("[저장] fs_scatter_v5.png")

---
## Phase 4: 주 추론 모형
`모형 A(SDI 단일) → 모형 B(F+S 분리, Wald 검정) → 건설업 → 업종 조절 효과`

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel

controls_ms  = 'ind_mfg + log_workers + elder_rate + foreign_rate + female_rate'
controls_con = 'log_workers + elder_rate + foreign_rate + female_rate'

def wald_test_bs_gt_bf(res):
    """β_S > β_F Wald 검정 (OLS 결과 객체에서 직접 계산)."""
    b_s  = res.params.get('S_score', np.nan)
    b_f  = res.params.get('F_score', np.nan)
    se_s = res.bse.get('S_score', np.nan)
    se_f = res.bse.get('F_score', np.nan)
    cov  = res.cov_params().loc['S_score','F_score'] if 'S_score' in res.params and 'F_score' in res.params else np.nan
    se_diff = np.sqrt(se_s**2 + se_f**2 - 2*cov)
    t_diff  = (b_s - b_f) / se_diff
    p_two   = 2 * (1 - stats.norm.cdf(abs(t_diff)))
    p_one   = p_two / 2
    return {'β_S': round(b_s,4), 'β_F': round(b_f,4),
            'β_S-β_F': round(b_s-b_f,4), 't': round(t_diff,4),
            'p(양측)': round(p_two,4), 'p(단측)': round(p_one,4),
            'β_S>β_F': b_s > b_f}

def reg_summary(res, label):
    """핵심 계수만 간결 출력."""
    print(f"\n{'─'*55}")
    print(f"[{label}]  n={int(res.nobs)}  R²={res.rsquared:.4f}")
    print(f"{'─'*55}")
    for var in ['SDI','F_score','S_score','ind_mfg','log_workers']:
        if var in res.params:
            b, p = res.params[var], res.pvalues[var]
            star = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else ''
            print(f"  {var:15s}: β={b:+.4f}  p={p:.4f}{star}")

In [ ]:
# 4-1. 모형 A — SDI 단일 지수 (OLS + 순서형 로지스틱)
print("=" * 55)
print("모형 A: ra_score ~ SDI + controls")
print("=" * 55)

# OLS
res_A_ms  = smf.ols(f'ra_score ~ SDI + {controls_ms}',  data=df_ms).fit()
res_A_con = smf.ols(f'ra_score ~ SDI + {controls_con}', data=df_con).fit()
reg_summary(res_A_ms,  '제조+서비스 OLS')
reg_summary(res_A_con, '건설 OLS')

# 순서형 로지스틱 (강건성)
def fit_ordered(df, controls):
    d = df[['ra_score','SDI'] + controls.replace(' ','').split('+')].dropna().copy()
    d['ra_score'] = d['ra_score'].astype(int)
    X = d[controls.replace(' ','').split('+') + ['SDI']]
    return OrderedModel(d['ra_score'], X, distr='logit').fit(method='bfgs', disp=False)

res_A_ms_ord  = fit_ordered(df_ms,  controls_ms)
res_A_con_ord = fit_ordered(df_con, controls_con)

for res, lab in [(res_A_ms_ord,'제조+서비스 순서형'), (res_A_con_ord,'건설 순서형')]:
    b = res.params.get('SDI', np.nan)
    p = res.pvalues.get('SDI', np.nan)
    star = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    print(f"  [{lab}] β_SDI={b:+.4f}  p={p:.4f} ({star})")

In [ ]:
# 4-2. 모형 B — F_score + S_score 분리 투입 + Wald 검정
print("=" * 55)
print("모형 B: ra_score ~ F_score + S_score + controls")
print("=" * 55)

res_B_ms  = smf.ols(f'ra_score ~ F_score + S_score + {controls_ms}',  data=df_ms).fit()
res_B_con = smf.ols(f'ra_score ~ F_score + S_score + {controls_con}', data=df_con).fit()
reg_summary(res_B_ms,  '제조+서비스 OLS')
reg_summary(res_B_con, '건설 OLS')

# Wald 검정
wald_ms  = wald_test_bs_gt_bf(res_B_ms)
wald_con = wald_test_bs_gt_bf(res_B_con)

print(f"\n[Wald 검정: H₀: β_S = β_F]")
print(f"  제조+서비스: β_S-β_F={wald_ms['β_S-β_F']:+.4f}  t={wald_ms['t']:.4f}  p(단측)={wald_ms['p(단측)']:.4f}  β_S>β_F: {wald_ms['β_S>β_F']}")
print(f"  건설:        β_S-β_F={wald_con['β_S-β_F']:+.4f}  t={wald_con['t']:.4f}  p(단측)={wald_con['p(단측)']:.4f}  β_S>β_F: {wald_con['β_S>β_F']}")

# 순서형 로지스틱 방향 일치 확인
def fit_ordered_B(df, controls):
    d = df[['ra_score','F_score','S_score'] + controls.replace(' ','').split('+')].dropna().copy()
    d['ra_score'] = d['ra_score'].astype(int)
    X = d[controls.replace(' ','').split('+') + ['F_score','S_score']]
    return OrderedModel(d['ra_score'], X, distr='logit').fit(method='bfgs', disp=False)

res_B_ms_ord  = fit_ordered_B(df_ms,  controls_ms)
res_B_con_ord = fit_ordered_B(df_con, controls_con)
print("\n[순서형 로지스틱 방향 확인]")
for res, lab in [(res_B_ms_ord,'제조+서비스'), (res_B_con_ord,'건설')]:
    bs = res.params.get('S_score', np.nan)
    bf = res.params.get('F_score', np.nan)
    print(f"  [{lab}] β_S={bs:+.4f}, β_F={bf:+.4f} → β_S>β_F: {bs>bf}")

In [ ]:
# 4-3. 가설 판정 + 결과 저장
print("\n" + "=" * 55)
print("가설 판정 종합")
print("=" * 55)

results = []
for label, res_A, res_B, wald in [
    ('제조+서비스', res_A_ms, res_B_ms, wald_ms),
    ('건설',        res_A_con, res_B_con, wald_con),
]:
    b_sdi = res_A.params.get('SDI', np.nan)
    p_sdi = res_A.pvalues.get('SDI', np.nan)
    h1 = (b_sdi > 0) and (p_sdi < 0.05)
    h2 = wald['β_S>β_F'] and (wald['p(단측)'] < 0.05)
    verdict = ('수준 1+2 지지 (최강)' if h1 and h2
               else '수준 2만 지지' if h2
               else '수준 1만 지지' if h1
               else '불지지')
    print(f"\n  [{label}]")
    print(f"    수준1 (β_SDI>0, p<0.05): β={b_sdi:+.4f} p={p_sdi:.4f} → {'✅' if h1 else '✗'}")
    print(f"    수준2 (β_S>β_F, p<0.05): diff={wald['β_S-β_F']:+.4f} p(단측)={wald['p(단측)']:.4f} → {'✅' if h2 else '✗'}")
    print(f"    → 종합 판정: {verdict}")
    results.append({'업종': label, 'β_SDI': b_sdi, 'p_SDI': p_sdi,
                    **wald, '판정': verdict})

pd.DataFrame(results).to_csv(OUTDIR + 'hypothesis_judgment_v5.csv', index=False, encoding='utf-8-sig')

# 4-4. 업종 조절 효과 (탐색적)
df_all = pd.concat([
    df_ms.assign(is_con=0),
    df_con.assign(is_con=1, ind_mfg=np.nan)
], ignore_index=True)

res_inter = smf.ols(
    'ra_score ~ SDI * is_con + log_workers + elder_rate + foreign_rate + female_rate',
    data=df_all
).fit()
b_inter = res_inter.params.get('SDI:is_con', np.nan)
p_inter = res_inter.pvalues.get('SDI:is_con', np.nan)
print(f"\n[업종 조절 효과 (탐색)] SDI×건설더미: β={b_inter:+.4f}  p={p_inter:.4f}")
print(f"  → 유의(p<0.05): {'✅ 건설업 이질성 통계적 지지' if p_inter<0.05 else '✗ 유의하지 않음'}")

---
## Phase 5: 강건성 검증
`Q17_13 재분류 → 50인 이상 표본 제한 → 결과 비교표`

In [ ]:
# 5-1. 민감도 1: Q17_13을 S로 이동
formal_groups_sens = {k: v for k, v in formal_groups_ms.items() if k != 'F-M8'}
subst_groups_sens  = {**subst_groups_ms, 'S-M8': ['Q17_13']}

df_ms['F_score_sens'] = compute_group_composite(df_ms, formal_groups_sens, '민감도 F')
df_ms['S_score_sens'] = compute_group_composite(df_ms, subst_groups_sens,  '민감도 S')
df_ms['SDI_sens']     = df_ms['S_score_sens'] - df_ms['F_score_sens']

res_sens_A = smf.ols(f'ra_score ~ SDI_sens + {controls_ms}',                         data=df_ms).fit()
res_sens_B = smf.ols(f'ra_score ~ F_score_sens + S_score_sens + {controls_ms}',      data=df_ms).fit()
wald_sens  = wald_test_bs_gt_bf(res_sens_B.rename(columns={'F_score_sens':'F_score','S_score_sens':'S_score'}) if hasattr(res_sens_B,'rename') else res_sens_B)

# wald_test 직접 계산
bs = res_sens_B.params.get('S_score_sens', np.nan)
bf = res_sens_B.params.get('F_score_sens', np.nan)
se_bs = res_sens_B.bse.get('S_score_sens', np.nan)
se_bf = res_sens_B.bse.get('F_score_sens', np.nan)
cov_sf = res_sens_B.cov_params().loc['S_score_sens','F_score_sens']
se_d = np.sqrt(se_bs**2 + se_bf**2 - 2*cov_sf)
t_d  = (bs - bf) / se_d
p_one_sens = (1 - stats.norm.cdf(abs(t_d)))  # 단측

print("─" * 55)
print("민감도 분석: Q17_13 → S로 이동")
print("─" * 55)
b_sens_sdi = res_sens_A.params.get('SDI_sens', np.nan)
p_sens_sdi = res_sens_A.pvalues.get('SDI_sens', np.nan)
print(f"  모형 A: β_SDI={b_sens_sdi:+.4f}  p={p_sens_sdi:.4f}")
print(f"  모형 B: β_S={bs:+.4f}, β_F={bf:+.4f}, diff={bs-bf:+.4f}, p(단측)={p_one_sens:.4f}")
print(f"  β_S>β_F: {bs>bf}")

# 5-2. 민감도 2: 50인 이상 사업장만
df_ms_50  = df_ms[df_ms['Q1_1'] >= 50].copy()
df_con_50 = df_con[df_con['Q4_1'] >= 50].copy()

res_50_ms  = smf.ols(f'ra_score ~ SDI + {controls_ms}',  data=df_ms_50).fit()
res_50_con = smf.ols(f'ra_score ~ SDI + {controls_con}', data=df_con_50).fit()

print(f"\n민감도 분석: 50인 이상 표본 제한")
print(f"  [제조+서비스 50인↑] n={int(res_50_ms.nobs)}  β_SDI={res_50_ms.params['SDI']:+.4f}  p={res_50_ms.pvalues['SDI']:.4f}")
print(f"  [건설 50인↑]        n={int(res_50_con.nobs)}  β_SDI={res_50_con.params['SDI']:+.4f}  p={res_50_con.pvalues['SDI']:.4f}")

In [ ]:
# 5-3. 강건성 비교표 저장
robust_rows = [
    {'분석': '기본 모형 A',        '업종': '제조+서비스', 'n': int(res_A_ms.nobs),
     'β_SDI': res_A_ms.params['SDI'], 'p': res_A_ms.pvalues['SDI'],
     'β_S-β_F': wald_ms['β_S-β_F'], 'p(단측)': wald_ms['p(단측)'], '비고': ''},
    {'분석': '기본 모형 A',        '업종': '건설',        'n': int(res_A_con.nobs),
     'β_SDI': res_A_con.params['SDI'], 'p': res_A_con.pvalues['SDI'],
     'β_S-β_F': wald_con['β_S-β_F'], 'p(단측)': wald_con['p(단측)'], '비고': ''},
    {'분석': '민감도: Q17_13→S',   '업종': '제조+서비스', 'n': int(res_sens_A.nobs),
     'β_SDI': b_sens_sdi, 'p': p_sens_sdi,
     'β_S-β_F': round(bs-bf,4), 'p(단측)': round(p_one_sens,4), '비고': 'Q17_13 S 이동'},
    {'분석': '민감도: 50인 이상',  '업종': '제조+서비스', 'n': int(res_50_ms.nobs),
     'β_SDI': res_50_ms.params['SDI'], 'p': res_50_ms.pvalues['SDI'],
     'β_S-β_F': np.nan, 'p(단측)': np.nan, '비고': 'Q1_1>=50'},
    {'분석': '민감도: 50인 이상',  '업종': '건설',        'n': int(res_50_con.nobs),
     'β_SDI': res_50_con.params['SDI'], 'p': res_50_con.pvalues['SDI'],
     'β_S-β_F': np.nan, 'p(단측)': np.nan, '비고': 'Q4_1>=50'},
]
robust_df = pd.DataFrame(robust_rows).round(4)
display(robust_df)
robust_df.to_csv(OUTDIR + 'robustness_v5.csv', index=False, encoding='utf-8-sig')
print("[저장] robustness_v5.csv")

---
## Phase 6: 건설업 이질성 분석
`경쟁 가설 C1(검력) / C2(측정 비동등) / C3(외생 결정 요인) 탐색`

In [ ]:
# 경쟁 가설 C1: SDI 분산이 작아 통계적 검력 부족
print("=" * 55)
print("경쟁 가설 C1: 건설업 SDI 분산 비교")
print("=" * 55)
sdi_std_ms  = df_ms['SDI'].std()
sdi_std_con = df_con['SDI'].std()
print(f"  제조+서비스 SDI σ: {sdi_std_ms:.4f}")
print(f"  건설         SDI σ: {sdi_std_con:.4f}")
print(f"  비율 (건설/제조서비스): {sdi_std_con/sdi_std_ms:.2f}")
print(f"  → C1 지지: {'✅ 건설업 분산 더 작음 (검력 낮을 수 있음)' if sdi_std_con < sdi_std_ms else '✗ 건설업 분산 더 큼'}")

# 경쟁 가설 C2: ra_score 분포 비동등 (건설업 바닥 효과)
print("\n" + "=" * 55)
print("경쟁 가설 C2: ra_score 분포 비교")
print("=" * 55)
for name, df in [('제조+서비스', df_ms), ('건설', df_con)]:
    vc = df['ra_score'].value_counts(normalize=True).sort_index()
    print(f"  [{name}]  미실시(0): {vc.get(0,0)*100:.1f}%  명목적(1): {vc.get(1,0)*100:.1f}%  실질적(2): {vc.get(2,0)*100:.1f}%")
print("  → C2: 건설업 ra_score=0 비율이 70% 이상이면 바닥 효과로 회귀 추정 불안정")

# ra_score 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, df) in zip(axes, [('제조+서비스', df_ms), ('건설', df_con)]):
    vc = df['ra_score'].value_counts(normalize=True).sort_index() * 100
    bars = ax.bar(['미실시(0)', '명목적(1)', '실질적(2)'],
                  [vc.get(i, 0) for i in [0, 1, 2]],
                  color=['#EF5350', '#FFA726', '#66BB6A'])
    for bar, val in zip(bars, [vc.get(i, 0) for i in [0, 1, 2]]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', fontsize=10)
    ax.set_title(f'{name}\nra_score 분포')
    ax.set_ylabel('%')
    ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig(OUTDIR + 'ra_score_dist_v5.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 경쟁 가설 C3: 외생 결정 요인 — 협력업체·원청 관계 변수 탐색
print("=" * 55)
print("경쟁 가설 C3: 건설업 외생 변수 탐색 (원청-하청 관계)")
print("=" * 55)

# 건설업 전용 변수 확인
c3_candidates = ['Q23_1','Q23_2','Q24_1','Q24_2','Q25_1','Q26_1']
avail_c3 = [c for c in c3_candidates if c in df_con.columns]
print(f"  건설업 협력·원청 관련 변수 탐색: {c3_candidates}")
print(f"  실제 존재 변수: {avail_c3}")

if avail_c3:
    # 대표 변수로 SDI 공변량 추가 후 효과 변화 확인
    formula_c3 = f'ra_score ~ SDI + {" + ".join(avail_c3[:3])} + {controls_con}'
    res_c3 = smf.ols(formula_c3, data=df_con).fit()
    b_sdi_c3 = res_c3.params.get('SDI', np.nan)
    p_sdi_c3 = res_c3.pvalues.get('SDI', np.nan)
    b_orig   = res_A_con.params.get('SDI', np.nan)
    print(f"\n  [C3 통제 후] β_SDI={b_sdi_c3:+.4f}  p={p_sdi_c3:.4f}")
    print(f"  [C3 통제 전] β_SDI={b_orig:+.4f}")
    print(f"  → 외생 변수 통제 후 SDI 효과 변화: {b_sdi_c3-b_orig:+.4f}")
else:
    print("  ℹ️  해당 변수 없음 — 코드북에서 건설업 원청 관련 변수명 확인 필요")

# 건설업 전용 변수 목록 출력 (탐색 보조)
all_con_cols = sorted(df_con.columns.tolist())
q23_plus = [c for c in all_con_cols if c.startswith(('Q23','Q24','Q25','Q26'))]
print(f"\n  건설업 Q23~Q26 계열 열명: {q23_plus[:20]}")

In [ ]:
# Phase 6 결과 요약 저장
c3_rows = []
for hyp, content, evidence in [
    ('C1 (검력 부족)',
     f'건설업 SDI σ={sdi_std_con:.4f} vs 제조+서비스 σ={sdi_std_ms:.4f}',
     '건설업 분산이 작으면 효과 탐지 어려움'),
    ('C2 (바닥 효과)',
     f'건설업 ra_score=0 비율 확인 필요',
     '70% 이상이면 회귀 추정 신뢰도 낮음'),
    ('C3 (외생 결정)',
     f'Q23~Q26 변수 존재 여부: {avail_c3 if avail_c3 else "없음"}',
     '원청-하청 관계 통제 후 SDI 효과 변화 관찰'),
]:
    c3_rows.append({'경쟁가설': hyp, '근거': content, '해석': evidence})

con_df = pd.DataFrame(c3_rows)
display(con_df)
con_df.to_csv(OUTDIR + 'construction_heterogeneity_v5.csv', index=False, encoding='utf-8-sig')
print("[저장] construction_heterogeneity_v5.csv")

print("\n" + "="*55)
print("Phase 6 완료 — 건설업 이질성 분석 결과")
print("="*55)
print("위 세 경쟁 가설 중 데이터가 가장 강하게 지지하는 설명을")
print("논문 토론 섹션에서 이론화할 것.")

# SDI v5 — From-Scratch 분석
> DESIGN.md 기반. 분석 전 결정사항 고정, 결과 보고 후 수정 없음.